# Monitoring, Hyperparameter Search & Advanced CNNs



In [ ]:
import random
from pathlib import Path

import torch
import torchvision
from torch import nn, optim
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, random_split

torch.manual_seed(1806)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# google colab data management
import os.path

try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    _home = 'gdrive/MyDrive/'
except ImportError:
    _home = '~'
finally:
    data_root = os.path.join(_home, '.pytorch')

print(data_root)

## Monitoring

Training a deep neural network with millions of parameters can cost quite some time. To make sure the network is training as expected,
it is crucial to monitor training progress in real time.

As a matter of fact, the `update` and `evaluate` functions
already implement some sort of ad hoc monitoring by providing the list of errors in a batch.
This list can be used to print the mean loss after every epoch
and can therefore be used to get an idea of how learning is progressing.
This specific implementation of monitoring the loss is not very flexible, however,
since it is not possible to access the information before the epoch has finished. Therefore, it is often necessary to log information more frequently. Moreover, additional information that show the model's training dynamics or help debugging a flawed model is frequently logged.

To thid end, various libraries exist. One of the most popular tools is [Weights & Biases (wandb)](https://wandb.ai/), which has largely replaced [Tensorboard](https://www.tensorflow.org/tensorboard) in recent years.

The standard pattern in most ML projects is straightforward:

1. **`wandb.init()`** — start a new run (with project name, config, etc.)
2. **`wandb.log()`** — log metrics (loss, accuracy, ...) during training
3. **`wandb.finish()`** — close the run when done

In this section, we integrate wandb directly into a `Trainer` class
so that every training run is automatically tracked.

The `Trainer` class provides the same functionality as the list that
you might have used in the current `update` and `evaluate` functions.
However, it also makes it possible to extend the functionality
of both functions without the need to interfere with existing code.

Note that there are libraries and frameworks out there that provide
(parts of) the functionality we will implement in what follows.
Two example frameworks that directly build on pytorch are
[pytorch-lightning](https://www.pytorchlightning.ai/)
and [pytorch ignite](https://pytorch.org/ignite/).

### Trainer with Monitoring

Below is a `Trainer` class that organises the training loop.
The goal of this exercise is to integrate wandb logging directly into the Trainer,
as is common practice in real ML projects.

 > Complete the `Trainer` class so that it:
 > - Initialises a wandb run in `__init__` (using the provided `wandb_config`).
 > - Logs the **per-batch loss** during `update()` as `"train/batch_loss"` at every gradient step.
 > - Logs the **average training and validation loss** after each epoch as `"train/loss"` and `"valid/loss"`.
 > - Calls `wandb.finish()` at the end of `train()`.
 >
 > The `train()` method should return a dict `{"train": ..., "valid": ...}` with the final average losses.

**Note:** Use `wandb.init(mode="offline")` in the config if you don't want to log to the cloud during development.

In [ ]:
# install wandb, if this throws an error
import wandb

In [ ]:
class Trainer:
    """ Class to organise learning and monitoring. """

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimiser: optim.Optimizer,
        wandb_config: dict = None,
    ):
        """
        Parameters
        ----------
        model : torch.nn.Module
            Neural Network that will be trained.
        criterion : torch.nn.Module
            Loss function to use for training.
        optimiser : torch.optim.Optimizer
            Optimiser for training.
        wandb_config : dict, optional
            Configuration dict passed to wandb.init().
            Useful keys: project, name, config, mode, ...
        """
        self.model = model
        self.criterion = criterion
        self.optimiser = optimiser

        self.epoch = 0
        self.global_step = 0

        # Initialize wandb
if wandb_config is not None:
            self.wandb_run = wandb.init(**wandb_config)
        else:
            self.wandb_run = None

    def state_dict(self):
        """ Current state of learning. """
        return {
            "model": self.model.state_dict(),
            "objective": self.criterion.state_dict(),
            "optimiser": self.optimiser.state_dict(),
            "num_epochs": self.epoch,
            "num_updates": self.global_step,
        }

    @property
    def device(self):
        """ Device of the (first) model parameters. """
        return next(self.model.parameters()).device

    @torch.no_grad()
    def evaluate(self, batches: DataLoader):
        """
        One epoch of evaluating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.eval()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

        avg_loss = sum(losses) / len(losses)
        return avg_loss

    @torch.enable_grad()
    def update(self, batches: DataLoader):
        """
        One epoch of updating the network.

        Parameters
        ----------
        batches : DataLoader
            An iterator over mini-batches of data to use for updating.
        tag : str, optional
            Identification tag for tracking loss values.

        Returns
        -------
        avg_loss : float
            The average loss over all mini-batches.
        """
        self.model.train()
        device = self.device

        losses = []
        for x, y in batches:
            x, y = x.to(device), y.to(device)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            losses.append(loss.item())

            self.optimiser.zero_grad()
            loss.backward()
            self.optimiser.step()

            # use wandb for logging train statistics during an epoch
self.global_step += 1
            if self.wandb_run is not None:
                wandb.log(
                    {"train/batch_loss": loss.item()},
                    step=self.global_step
                )


        avg_loss = sum(losses) / len(losses)
        return avg_loss

    def train(self, train_batches, valid_batches=None, num_epochs: int = 1):
        """
        Train the network for multiple epochs.

        Parameters
        ----------
        train_batches : DataLoader
            The training data for updating the network.
        valid_batches : DataLoader, optional
            The validation data for estimating the generalisation performance.
        num_epochs : int, optional
            The number of epochs to train.

        Returns
        -------
        results : dict
            The average loss estimates after `num_epochs` epochs.

        """
        if valid_batches is None:
            valid_batches = ()

        # implement the whole training loop
        # log metrics
for epoch in range(num_epochs):
          self.epoch = epoch
          train_loss = self.update(train_batches)

        if valid_batches:
            valid_loss = self.evaluate(valid_batches)
        else:
            valid_loss = None

        if self.wandb_run is not None:
                log_dict = {"train/loss": train_loss}
                if valid_loss is not None:
                    log_dict["valid/loss"] = valid_loss

                wandb.log(log_dict, step=self.global_step)

        # finish wandb run
        if self.wandb_run is not None:
            wandb.finish()

        return {"train": train_loss, "valid": valid_loss}

In [ ]:
# Sanity check
# Build a tiny dataset and loader for testing
from torchvision import transforms

_dummy_data = torch.randn(128, 1, 8, 8)
_dummy_labels = torch.randint(0, 3, (128,))
_dummy_ds = torch.utils.data.TensorDataset(_dummy_data, _dummy_labels)
loader = DataLoader(_dummy_ds, batch_size=32)

conv_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 3),
)


In [ ]:
# Sanity check
# basic training loop
trainer = Trainer(
    model=conv_net.to(device),
    criterion=nn.CrossEntropyLoss(),
    optimiser=optim.Adam(conv_net.parameters(), lr=1e-2),
    wandb_config={"project": "dl-test", "mode": "offline", "name": "sanity-check"},
)
results = trainer.train(loader, loader, num_epochs=3)

assert "train" in results, "Could not find training loss in results"
assert "valid" in results, "Could not find validation loss in results"
assert isinstance(results["train"], float), f"Expected float, got {type(results['train'])}"
assert isinstance(results["valid"], float), f"Expected float, got {type(results['valid'])}"
assert (trainer.epoch in (1, 2)), f"Expected 2 or 3 epochs, got {trainer.epoch}"
assert trainer.global_step > 0, f"Global_step not updated"

## Hyperparameter Search

Finding good hyperparameters for a model is a general problem in machine learning (or even statistics).
However, neural networks are (in)famous for their large number of hyperparameters.
To list a few: learning rate, batch size, epochs, pre-processing, layer count, neurons for each layer,
activation function, initialisation, normalisation, layer type, skip connections, regularisation, ...
Moreover, it is often not possible to theoretically justify a particular choice for a hyperparameter.
E.g. there is no way to tell whether $N$ or $N + 1$ neurons in a layer would be better, without trying it out.
Therefore, hyperparameter search for neural networks is an especially tricky problem to solve.

###### Manual Search

The most straightforward approach to finding good hyperparameters is to just
try out *reasonable* combinations of hyperparameters and pick the best model (using e.g. the validation set).
The first problem with this approach is that it requires a gut feeling as to what *reasonable* combinations are.
Moreover, it is often unclear how different hyperparameters interact with each other,
which can make an irrelevant hyperparameter look more important than it actually is or vice versa.
Finally, manual hyperparameter search is time consuming, since it is generally not possible to automate.

###### Grid Search

Getting a feeling for combinations of hyperparameters is often much harder than for individual hyperparameters.
The idea of grid search is to get a set of *reasonable* values for each hyperparameter individually
and organise these sets in a grid that represents all possible combinations of these values.
Each combinations of hyperparameters in the grid can then be run simultaneously,
assuming that so much hardware is available, which can speed up the search significantly.

###### Random Search

Since there are plenty of hyperparameters and each hyperparameters can have multiple *reasonable* values,
it is often not feasible to try out every possible combination in the grid.
On top of that, most of the models will be thrown away anyway because only the best model is of interest,
even though they might achieve similar performance.
The idea of random search is to randomly sample configurations, rather than choosing from pre-defined choices.
This can be interpreted as setting up an infinite grid and trying only a few --- rather than all --- possibilities.
Under the assumption that there are a lot of configurations with similarly good performance as the best model,
this should provide a model that performs very good with high probability for a fraction of the compute.

###### Bayesian Optimisation

Rather than picking configurations completely at random,
it is also possible to guide the random search.
This is essentially the premise of Bayesian optimisation:
sample inputs and evaluate the objective to find which parameters are likely to give good performance.

Bayesian optimisation uses a function approximator for the objective
and what is known as an *acquisition* function.
The function approximator, or *surrogate*,
has to be able to model a distribution over function values, e.g. a Gaussian Process.
The acquisition function then uses these distributions
to find where the largest improvements can be made, e.g. using the cdf.
For a more elaborate explanation of Bayesian optimisation,
see e.g. [this tutorial](https://arxiv.org/abs/1807.02811)

This approach is less parallellisable than grid or random search,
since it uses the information from previous runs to find good sampling regions.
However, often there are more configurations to be tried out than there are computing devices
and it is still possible to sample multiple configurations at each step with Bayesian Optimisation.
Also consider [this paper](https://papers.nips.cc/paper/4522-practical-bayesian-optimization-of-machine-learning-algorithms) in this regard.

###### Neural Architecture Search

Instead of using Bayesian optimisation,
the problem of hyperparameter search can also be tackled by other optimisation algorithms.
This approach is also known as *Neural Architecture Search* (NAS).
There are different optimisation strategies that can be used for NAS,
but the most common are evolutionary algorithms and (deep) reinforcement learning.
Consider reading [this survey](http://jmlr.org/papers/v20/18-598.html)
to get an overview of how NAS can be used to construct neural networks.

## Efficient CNNs

In recent times CNNs have become more computationally efficient. Traditional convolutional layers apply filters across the entire depth of the input volume, mixing all the input channels to produce a single output channel. Depthwise separable convolutions, introduced as a key innovation in architectures like Xception, are a more efficient variant of the standard convolution operation. This process is divided into two layers: the depthwise convolution and the pointwise convolution. In the depthwise convolution, a single filter is applied per input channel, which significantly reduces the computational cost. Following this, a 1x1 convolution (pointwise convolution) is applied to combine the outputs of the depthwise layer, creating a new set of feature maps. This approach drastically reduces the number of parameters and computations, making the network more efficient and faster, which is especially beneficial for mobile and embedded devices.

<img src="https://www.researchgate.net/publication/358585116/figure/fig1/AS:1127546112487425@1645839350616/Depthwise-separable-convolutions.png" />

Squeeze-and-Excitation layers introduce an additional level of adaptivity in CNNs, enabling the network to perform dynamic channel-wise feature recalibration. Squeeze-and-Exitation blocks are usually executed after a convolutional layer or block
and before the residual connection by a series of relatively inexpensive computations

1. A three dimensional input consisting of different channels and the two spati l
dimensions is compressed into one dimension by global aver ge pooling. As a res lt
the spatial information is squeezed into one descriptor per channel.
2. The squeezed data is transformed by a two layer feed-forward neural network.  fter
the first linear layer ReLU is used as activation functi n and after the se ond a
sigmoid function is applied. This normalizes the output between 0 and 1 and can be
interpreted as the significance per channel.
3. The result is used to scale the input of the Squeeze-and-Exitation block by an element-
wise multiplication.

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*bmObF5Tibc58iE9iOu327w.png" />



### Create an Efficient CNN

Today, neural networks frequently have millions or billions of parameters. However, CNNs have become more computationally efficient over the years. How far can you get with a limited amount of compute?

> Create an efficient CNN with less than 30.000 parameters.
> Use at least one depthwise separable or groupwise convolution or apply at least one squeeze-and-exitation layer after a convolution.

Hint: Skip-connections and Normalization layers are frequently used to stabilize the training behavior of deep CNNs.

In [ ]:
class EfficientCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
super().__init__()

        # Depthwise separable conv block
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.dwconv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16)
        self.pwconv2 = nn.Conv2d(16, 32, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.dwconv3 = nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32)
        self.pwconv3 = nn.Conv2d(32, 64, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Linear(64, num_classes)

        self.act = nn.ReLU()
    def forward(self, x):
x = self.act(self.bn1(self.conv1(x)))

        x = self.act(self.bn2(self.pwconv2(self.dwconv2(x))))
        x = self.pool(x)

        x = self.act(self.bn3(self.pwconv3(self.dwconv3(x))))
        x = self.pool(x)

        x = self.gap(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        return x






In [ ]:
# sanity-check
model = EfficientCNN(in_channels=3, num_classes=10)
model(torch.zeros((1, 3, 32, 32)))
print("number of parameters: ", sum([p.numel() for p in model.parameters()]))

### Training

In order to get a feeling for hyperparameter search, you have to try it out on some example. You can use the monitoring tools from previous exercises to log performance and get a feeling for which hyperparameters work well.

> Train your EfficientCNN on CIFAR10 using the Trainer class. Use hyperparameter search for the learning rate, optimizer and maybe even the model architecture to get a CrossEntropyLoss < 1.5 within 10 epochs of training with a fixed batch size of 1024.

In [ ]:
import torchvision.transforms as transforms

train_dataset = torchvision.datasets.CIFAR10(
    data_root, train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = torchvision.datasets.CIFAR10(
    data_root, train=False, transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=2)


configs = [
    {"opt": "adam", "lr": 1e-3},
    #{"opt": "adam", "lr": 3e-4},
    #{"opt": "adam", "lr": 1e-4},
    #{"opt": "sgd", "lr": 0.1},
    #{"opt": "sgd", "lr": 0.05},
]

all_results = []

for cfg in configs:
    print("\nRunning:", cfg)

    model = EfficientCNN(in_channels=3, num_classes=10)  # همون مدل خودت

    if cfg["opt"] == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9)

    trainer = Trainer(
        model,
        nn.CrossEntropyLoss(),
        optimizer,
        wandb_config={
            "project": "dl-a3",
            "mode": "offline",
            "config": cfg
        }
    )

    result = trainer.train(train_loader, test_loader, num_epochs=10)
    all_results.append((cfg, result))

print("\nFinal Results:")
for cfg, res in all_results:
    print(cfg, res)

## Beyond Convolutions: Alternative Vision Architectures

Convolutional neural networks have dominated computer vision for over a decade.
However, several alternative architectures have emerged that challenge
the assumption that convolutions are necessary for visual understanding.

###### Vision Transformers (ViT)

The [Vision Transformer (ViT)](https://arxiv.org/abs/2010.11929) applies the Transformer architecture,
originally developed for NLP, directly to image classification.
An image is split into fixed-size patches (e.g. 16×16), each patch is linearly embedded,
and the resulting sequence of patch embeddings is processed by a standard Transformer encoder.
A special `[CLS]` token is prepended to the sequence and its final representation is used for classification.
ViTs achieve state-of-the-art results but typically require very large datasets
or extensive pre-training to outperform CNNs.

###### MLP-Mixer

The [MLP-Mixer](https://arxiv.org/abs/2105.01601) takes a more radical approach:
it uses **only MLPs** — no convolutions, no self-attention.
Like ViT, the image is divided into non-overlapping patches that are linearly projected.
The architecture then alternates between two types of MLP layers:

1. **Token-mixing MLPs** — applied across the spatial (patch) dimension,
   allowing communication between different spatial locations.
   All channels of one patch are mixed with all channels of every other patch.
2. **Channel-mixing MLPs** — applied independently to each patch,
   mixing information across the feature/channel dimension.

Each mixer layer applies layer normalisation, followed by the MLP, and uses a residual connection:

$$U = X + W_2 \, \sigma(W_1 \, \text{LayerNorm}(X)^T)^T \quad \text{(token-mixing)}$$
$$Y = U + W_4 \, \sigma(W_3 \, \text{LayerNorm}(U)) \quad \text{(channel-mixing)}$$

Given the lack of inductive bias for image processing, this design is surprisingly competitive.

### Implement an MLP-Mixer

Now it is your turn to implement a vision architecture that uses **no convolutions and no attention**.

 > Implement an `MLPMixer` model for CIFAR-10 (32×32 RGB images, 10 classes) with **fewer than 30,000 parameters**.
 >
 > Your implementation must include:
 > - A **patch embedding** layer that splits the image into non-overlapping patches and projects them.
 > - At least **2 mixer layers**, each consisting of a token-mixing MLP and a channel-mixing MLP.
 > - **Residual connections** and **layer normalisation** in each mixer layer.
 > - A **global average pooling** + linear classification head.

**Hint:** With 32×32 images, a patch size of 8 gives you 16 patches.
Choose the hidden dimension and MLP expansion factors carefully to stay under the parameter budget.
GELU is the standard activation for mixer MLPs.


In [ ]:
class MixerBlock(nn.Module):
    """A single Mixer layer: token-mixing followed by channel-mixing."""

    def __init__(self, num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim):
        super().__init__()
        # Token-mixing: operates across the patch (spatial) dimension
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_mlp_dim),
            nn.GELU(),
            nn.Linear(token_mlp_dim, num_patches),
        )
        # Channel-mixing: operates across the channel/feature dimension
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(hidden_dim, channel_mlp_dim),
            nn.GELU(),
            nn.Linear(channel_mlp_dim, hidden_dim),
        )

    def forward(self, x):
        # x shape: (batch, num_patches, hidden_dim)
#Token mixing
        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.token_mlp(y)
        y = y.transpose(1, 2)
        x = x + y

        #Channel mixing
        y = self.norm2(x)
        y = self.channel_mlp(y)
        x = x + y

        return x


class MLPMixer(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size  # flattened patch

        # Patch embedding: project each flattened patch to hidden_dim
        self.patch_embed = nn.Linear(patch_dim, hidden_dim)

        # Mixer layers
        self.mixer_layers = nn.Sequential(*[
            MixerBlock(num_patches, hidden_dim, token_mlp_dim, channel_mlp_dim)
            for _ in range(num_layers)
        ])

        # Classification head
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
#  Patch extraction
        B, C, H, W = x.shape
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.contiguous().view(B, C, -1, p, p)
        x = x.permute(0, 2, 1, 3, 4)
        x = x.reshape(B, -1, C * p * p)

        #  Patch embedding
        x = self.patch_embed(x)

        # Mixer layers
        x = self.mixer_layers(x)

        # Global pooling
        x = x.mean(dim=1)

        #  Classification
        x = self.norm(x)
        x = self.head(x)

        return x


In [ ]:
# sanity-check
mixer = MLPMixer(in_channels=3, num_classes=10, image_size=32, patch_size=8,
                 hidden_dim=32, num_layers=2, token_mlp_dim=64, channel_mlp_dim=64)
out = mixer(torch.zeros((1, 3, 32, 32)))
print(f"Output shape: {out.shape}")
num_params = sum(p.numel() for p in mixer.parameters())
print(f"Number of parameters: {num_params}")


### Train the MLP-Mixer

 > Train your `MLPMixer` on CIFAR-10 using the `Trainer` class.
 > Use wandb to log and compare runs.
 > Achieve a `CrossEntropyLoss < 1.5` within 10 epochs.

**Hint:** MLP-Mixers can be sensitive to the learning rate.
Try Adam or AdamW with learning rates in the range 1e-3 to 5e-3.

In [ ]:
# TODO: Train your MLPMixer and tune hyperparameters
import torchvision.transforms as transforms

# dataset
train_dataset = torchvision.datasets.CIFAR10(
    data_root, train=True, transform=transforms.ToTensor(), download=True
)
test_dataset = torchvision.datasets.CIFAR10(
    data_root, train=False, transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=2)


configs = [
    {"lr": 1e-3, "opt": "adam"},
   # {"lr": 3e-3, "opt": "adam"},
    #{"lr": 5e-3, "opt": "adam"},
   # {"lr": 1e-3, "opt": "adamw"},
]

results_all = []

for cfg in configs:
    print("\nRunning:", cfg)

    model = MLPMixer()  # همون مدل خودت

    if cfg["opt"] == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])

    trainer = Trainer(
        model,
        nn.CrossEntropyLoss(),
        optimizer,
        wandb_config={
            "project": "mlp-mixer-a3",
            "mode": "offline",
            "config": cfg
        }
    )

    results = trainer.train(train_loader, test_loader, num_epochs=10)
    results_all.append((cfg, results))


print("\nFinal Results:")
for cfg, res in results_all:
    print(cfg, res)

### Conclusions
The questions below should help you to reflect on the experiments. Enter your answer directly in this cell or create a new cell.
 - If you compare CNNs and MLP-mixer: What is the more universal architecture? Which inductive biases do they have?
 - You might have noticed that the difference of training and test error is not the same in the efficient CNN as in the MLP-mixer. What could be the reason? Which architecture would you use for image classification?

1. CNN vs MLP-Mixer: Universality and Inductive Bias

Convolutional Neural Networks (CNNs) are less universal but more specialised for image data.
They incorporate strong inductive biases, such as:
	•	Locality: nearby pixels are related
	•	Translation invariance: features can appear anywhere
	•	Weight sharing: same filters across the image

These biases make CNNs highly efficient for image tasks, especially with limited data.

In contrast, the MLP-Mixer is a more universal architecture, since it does not assume any spatial structure in the data.
It relies on:
	•	Token-mixing MLPs to learn spatial relationships
	•	Channel-mixing MLPs to learn feature interactions

However, because it lacks strong inductive biases, it typically requires more data or careful tuning to achieve similar performance.



2. Difference Between Training and Test Error

In our experiments, the difference between training and validation error was not the same for CNNs and the MLP-Mixer.

A likely reason is that:
	•	CNNs, due to their inductive biases, generalise better and are less prone to overfitting on small datasets like CIFAR-10
	•	The MLP-Mixer, being more flexible and less constrained, may either:
	•	underfit (if capacity is too small), or
	•	overfit (if not properly regularised)

Additionally, CNNs extract structured spatial features more efficiently, while the MLP-Mixer must learn these relationships from scratch, which can lead to different training dynamics.



3. Which Architecture Would I Use?

For image classification, especially on relatively small datasets like CIFAR-10, I would choose a CNN-based architecture.

Reason:
	•	Better performance with limited data
	•	Faster convergence
	•	Strong inductive biases aligned with image structure

However, for very large datasets or when scalability and flexibility are important, architectures like the MLP-Mixer (or Vision Transformers) become more competitive.
